# Karpathy's GPT from scratch

Following [Let's build GPT: from scratch, in code, spelled out.](https://www.youtube.com/watch?v=kCc8FmEb1nY)

For each character, generate a prediction and hidden state at each step by feeding its *previous* hidden state in each next step. Take the final prediction to be the output, which class the word belongs to.

The task is to train on thousands of last names from 18 different languages and predict which language a name is based on the spelling.

In [1]:
import string
import unicodedata
import pandas as pd
import torch
from pathlib import Path

In [2]:
%load_ext watermark

In [3]:
# Check if CUDA is available
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


# Data inspection

In [4]:
PROJECT_DATA = "/workspaces/transformer-mentor/project_data/"
with open(Path(PROJECT_DATA) / "input.txt", "r", encoding="utf-8") as file:
    text = file.read()

In [5]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  10045


In [14]:
print(text[:200])

Planet Telex
You can force it but it will not come
You can taste it but it will not form
You can crush it but it's always here
You can crush it but it's always near
Chasing you home, saying
Everything


In [7]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("all the unique characters:", "".join(chars))
print(f"vocab size: {vocab_size}")

all the unique characters: 
 '(),-.06?ABCDEFGHIJKLMNOPRSTWYabcdefghijklmnoprstuvwxyz
vocab size: 57


In [11]:
# tokenize, create a mapping from characters to integers and vice versa
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# encoder: take a string, output a list of integers
encode = lambda s: [stoi[c] for c in s]

# decoder: take a list of integers, output a string
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hii there'))
print(decode(encode('hii there')))

[39, 40, 40, 1, 50, 39, 36, 48, 36]
hii there


Modern tokenizers include Google SentencePiece, OpenAI tiktoken

In [13]:
# tokenize the text dataset
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:200])

torch.Size([10045]) torch.int64
tensor([26, 43, 32, 45, 36, 50,  1, 29, 36, 43, 36, 54,  0, 31, 46, 51,  1, 34,
        32, 45,  1, 37, 46, 48, 34, 36,  1, 40, 50,  1, 33, 51, 50,  1, 40, 50,
         1, 53, 40, 43, 43,  1, 45, 46, 50,  1, 34, 46, 44, 36,  0, 31, 46, 51,
         1, 34, 32, 45,  1, 50, 32, 49, 50, 36,  1, 40, 50,  1, 33, 51, 50,  1,
        40, 50,  1, 53, 40, 43, 43,  1, 45, 46, 50,  1, 37, 46, 48, 44,  0, 31,
        46, 51,  1, 34, 32, 45,  1, 34, 48, 51, 49, 39,  1, 40, 50,  1, 33, 51,
        50,  1, 40, 50,  2, 49,  1, 32, 43, 53, 32, 55, 49,  1, 39, 36, 48, 36,
         0, 31, 46, 51,  1, 34, 32, 45,  1, 34, 48, 51, 49, 39,  1, 40, 50,  1,
        33, 51, 50,  1, 40, 50,  2, 49,  1, 32, 43, 53, 32, 55, 49,  1, 45, 36,
        32, 48,  0, 13, 39, 32, 49, 40, 45, 38,  1, 55, 46, 51,  1, 39, 46, 44,
        36,  5,  1, 49, 32, 55, 40, 45, 38,  0, 15, 52, 36, 48, 55, 50, 39, 40,
        45, 38])


In [3]:
%watermark

Last updated: 2025-08-06T23:14:29.951880+00:00

Python implementation: CPython
Python version       : 3.12.11
IPython version      : 9.4.0

Compiler    : GCC 12.2.0
OS          : Linux
Release     : 6.10.14-linuxkit
Machine     : aarch64
Processor   : 
CPU cores   : 7
Architecture: 64bit



In [4]:
%watermark -iv

In [ ]:
# test